例えば、大学前のアパートを入力とし、そこから20分で移動できる場所を取得したい。以下の合計を求めることになる
・入力から徒歩20分のエリア
・入力から20分以内に到達可能な交通機関から(20分-駅までの時間)分で到着できる別の駅とそこから(20分-(駅までの時間+別の駅までの乗車時間))分以内で到達できるエリア、再帰の可能性あり  
手順
1. 20分以内の徒歩エリアを求める
2. 20分以内で到達できる徒歩での乗り換えが必要ない交通機関を求める(以下ダイレクトと呼称)
3. (ダイレクトまでの徒歩時間と、ダイレクトまでの乗車時間を引いた)時間以内の、ダイレクトからの徒歩エリアを取得
4. 2, 3を繰り返す.
5. すべてを結合

3,4を作成

In [1]:
from dotenv import load_dotenv

from engine import search_nearby_walking_distance_stations, get_stations_contain_area, get_same_line_or_route_stations, \
    get_route_yahoo_transit, get_same_line_or_route_stations_with_time
from engine.mapbox import MapBoxApi, IsochroneProfile, concat_isochrones
import os
from engine.bus import *
from engine.train import *


# データの読み込み
dataset: dict[TransitType, list[Station]] = {
    TransitType.BUS: load_stop_data("../dataset/busstops/kanagawa/P11-22_14.geojson"),
    TransitType.TRAIN: load_station_data("../dataset/stations/N02-20_Station.geojson"),
}
load_dotenv()
mapbox_api = MapBoxApi(os.getenv("MAPBOX_API_TOKEN"))

In [19]:
def search_reachable_area(
        api: MapBoxApi, dataset: dict[TransitType, list[Station]],
        start_point: Coordinate, time_limit: int, transit_types: list[TransitType]) -> list[dict]:
    # reachable_areas = {tl: [] for tl in time_limits} # { int(tl1): [] }で初期化
    reachable_area:list[dict] = []

    # (1) 入力位置の周辺
    walking_distance_area = api.get_isochrone(
        prof=IsochroneProfile.Walking,
        coordinate=start_point,
        contours_minutes=[time_limit]
    )
    print("[1] Get Walking Distance Isochrone: Done")
    
    # 徒歩圏内に含まれる交通機関一覧
    _stations = get_stations_contain_area(
        dataset=dataset,
        isochrone=walking_distance_area,
        transit_types=transit_types,
    )
    print("[1] Get Stations within Isochrone: Done")
    
    # 各交通機関まで徒歩でかかる時間を計算
    print("[1] Get Station Travel time: Running...")
    stations_with_travel_time: list[tuple[int, Station]] = []
    for station in _stations:
        travel_time = api.get_walking_travel_time(start_point, station.geometry.calc_mean())
        stations_with_travel_time.append(
            (travel_time, station)
        )
        # print(f"  - {station.name}: {travel_time}mins")
    print("[1] Get Station Travel time: Done")
    
    # (2) 徒歩圏内の交通機関を使って時間以内に行ける別の駅を表示する
    print("[2] Get Stations from Result of [1]: Running...")
    reachable_stations: list[tuple[int, Station]] = []
    for walking_travel_time, station in stations_with_travel_time:
        # (最大移動時間time_limit-各交通機関への移動時間)分以内に辿り着ける同じ路線の駅を取得
        same_line_stations_with_travel_time = get_same_line_or_route_stations_with_time(station, dataset, time_limit-walking_travel_time)
        for transit_travel_time, same_line_station in same_line_stations_with_travel_time:
            reachable_stations.append(
                # 交通機関までの移動時間とそこから交通機関をつかった時間
                (walking_travel_time+transit_travel_time, same_line_station)
            )
    print("[2] Get Stations from Result of [1]: Done")

    # 移動先の交通機関からtime_limit以内に行ける範囲
    print("[2] Isochrone from Result of [2]: Running...")
    for travel_time, reachable_station in reachable_stations:
        remaining_time = time_limit-travel_time
        if 60 >= remaining_time >= 1:
            isochrone = api.get_isochrone(
                prof=IsochroneProfile.Walking,
                coordinate=reachable_station.geometry.calc_mean(),
                contours_minutes=[remaining_time]
            )
            reachable_area.append(
                isochrone
            )
    print("[2] Isochrone from Result of [2]: Done")
    
    return reachable_area
    


In [18]:
# 厚木市役所を入力とする.
input_coordinate = Coordinate(Lat=35.4429973, Lng=139.3611488)
# 上記まで30分で行ける範囲を検索する
input_time_limit = 30

reachable_area = search_reachable_area(
    api=mapbox_api, dataset=dataset,
    start_point=input_coordinate,
    time_limit=input_time_limit,
    transit_types=[TransitType.BUS, TransitType.TRAIN],
)

[1] Get Walking Distance Isochrone: Done
[1] Get Stations within Isochrone: Done
[1] Get Station Travel time: Running...
[1] Get Station Travel time: Done
[2] Get Stations from Result of [1]: Running...
!! insert ERROR !!
UNIQUE constraint failed: routes.is_bus_route, routes.from_, routes.to_, routes.time_required, routes.transfer, routes.fare, routes.distance
InsertRouteReq(is_bus_route=True, from_=Station(transit_type=<TransitType.BUS: 2>, name='あつぎ大通り', management_groups=['神奈川中央交通（株）'], line_routes=['厚01', '厚02', '厚03', '厚04', '厚05', '厚06', '厚07', '厚08', '厚09', '厚10', '厚11', '厚12', '厚14', '厚25', '厚26', '厚32', '厚33', '厚34', '厚38', '厚39', '厚46', '厚47', '厚48', '厚66', '厚67', '厚80', '厚81', '厚89', '厚94', '厚95', '厚97', '厚108'], geometry=Geometry(Type='Point', Coordinates=[Coordinate(Lng=139.36235307148507, Lat=35.441651435648886)]), raw_feature={'type': 'Feature', 'properties': {'P11_001': 'あつぎ大通り', 'P11_002': '神奈川中央交通（株）', 'P11_003_01': '厚01,厚02,厚03,厚04,厚05,厚06,厚07,厚08,厚09,厚10,厚11,厚12,厚14

In [22]:
import folium
from backend.engine.mapbox import concat_isochrones
m = folium.Map(
    location=[35.4861002, 139.3399782],
    zoom_start=14,
    tiles="https://cyberjapandata.gsi.go.jp/xyz/std/{z}/{x}/{y}.png",
    attr=f"出典: 国土地理院ウェブサイト・地理院タイル・標準地図 {'(C) MAPBOX'}",
)

# print(reachable_area)

# print(reachable_area[60])
# folium.GeoJson(reachable_area).add_to(m)
for isochrone in reachable_area:
    folium.GeoJson(isochrone).add_to(m)
    
m